
# TLC Trip Record Data

## Yellow Taxi Trip Records 2026 (January - )


https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page


https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf

In [1]:
import pandas as pd
import pyarrow
import numpy as np

### Data Cleaning Notes

renamed several columns to improve clarity. 
- tpep_pickup_datetime: pickup_datetime 
- tpep_dropoff_datetime" : dropoff_datetime
- extra : extra_charges
- trip_distance : trip_distance_miles

replaced strange unusable 0 values in columns where that's impossible, with Nan.
- passenger count

I noticed impossibly high distances recorded so i grabbed the largeest and compared them to the fares and noticed no consitency proving that that this column is faulty and contains errors I believe came from data collection. My theory is that these values werent necesarrly entered wrong but that the taximeter within that cab at that time or was faulty and recording incorrect information. 

I plan to check and see if i can find any other overlap between these outliers including 


of this dataset of over 3 million taxi trips only 162 have trips have trips recorded to be over 100 miles and 620 over 50 miles. 

- PULocationID	DOLocationID and fare_amount are a better measure of the length of the ride (taxi zone look up csv on tlc site)


### Data Analysis
 ideas
- check for overlap on vendor id or other identifers like time of year for any specifc types of errors(like the passenger count or trp distance outliers)
- Which day generated the highest total revenue?
- How much money is generated by each payment type?
- What are the busiest pickup hours?
- Do larger groups travel farther?
- Does passenger count affect tip percentage?

visualizations 
- What percentage of trips use each payment method?
- Revenue by Hour


In [29]:

file = "/Users/kamari/Documents/project_info/yellow_tripdata_2026-JAN.parquet"

zone_lookup= pd.read_csv("/Users/kamari/Documents/project_info/taxi_zone_lookup.csv")

taxi_zones = pd.DataFrame(zone_lookup)


data = pd.read_parquet(file)
df = pd.DataFrame(data)

df.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

In [140]:
yellow_taxi = df.copy()

In [141]:
yellow_taxi.rename(columns={
    "tpep_pickup_datetime": "pickup_datetime" , 
    "tpep_dropoff_datetime" : "dropoff_datetime",
    "extra" : "extra_charges",
    "trip_distance" : "trip_distance_miles",
}, inplace=True)


In [142]:
yellow_taxi.drop(columns=['store_and_fwd_flag', 'improvement_surcharge'], inplace=True)

In [143]:
yellow_taxi['passenger_count'] = yellow_taxi['passenger_count'].replace(0, np.nan)
yellow_taxi


,VendorID,pickup_datetime,dropoff_datetime,passenger_count,trip_distance_miles,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra_charges,mta_tax,tip_amount,tolls_amount,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,239,238,1,7.20,1.00,0.5,3.66,0.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,NaN,0.90,1.0,163,162,2,7.90,4.25,0.5,0.00,0.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,NaN,1.40,1.0,43,237,1,10.70,4.25,0.5,2.50,0.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,142,209,1,38.70,1.00,0.5,11.11,0.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,NaN,2.16,1.0,88,144,1,13.50,1.00,0.5,3.85,0.0,23.10,2.5,0.0,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724884,2,2026-01-31 23:26:00,2026-01-31 23:39:16,NaN,1.62,NaN,237,161,0,17.09,0.00,0.5,0.00,0.0,21.84,NaN,NaN,0.75
3724885,2,2026-01-31 23:33:53,2026-01-31 23:34:07,NaN,0.00,NaN,42,42,0,23.19,0.00,0.5,0.00,0.0,24.69,NaN,NaN,0.00
3724886,2,2026-01-31 23:40:23,2026-01-31 23:56:10,NaN,6.84,NaN,137,69,0,29.21,0.00,0.5,0.00,0.0,33.96,NaN,NaN,0.75
3724887,2,2026-01-31 23:10:21,2026-01-31 23:20:00,NaN,1.53,NaN,137,162,0,19.19,0.00,0.5,0.00,0.0,23.94,NaN,NaN,0.75


In [115]:
yellow_taxi.isnull().sum()

VendorID                      0
pickup_datetime               0
dropoff_datetime              0
passenger_count         1102845
trip_distance_miles           0
RatecodeID              1088058
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra_charges                 0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
total_amount                  0
congestion_surcharge    1088058
Airport_fee             1088058
cbd_congestion_fee            0
dtype: int64

In [144]:
yellow_taxi.loc[
    yellow_taxi["trip_distance_miles"].nlargest(20).index,
    ["VendorID", "trip_distance_miles", "fare_amount"]
]

,VendorID,trip_distance_miles,fare_amount
3524923,2,269097.48,12.19
3632367,2,237567.19,72.99
3285750,2,236216.69,13.58
3131676,2,236183.06,22.75
2704619,2,221544.91,31.21
2688274,2,221436.73,27.61
3131406,2,196490.48,33.59
3571687,2,192081.68,60.20
2820951,2,184908.33,12.18
3499856,2,183363.75,71.01


In [10]:
(yellow_taxi["trip_distance_miles"] > 50).sum()

(yellow_taxi["trip_distance_miles"] > 100).sum()

np.int64(162)

In [117]:
yellow_taxi[["PULocationID","DOLocationID"]]




,PULocationID,DOLocationID
0,239,238
1,163,162
2,43,237
3,142,209
4,88,144
...,...,...
3724884,237,161
3724885,42,42
3724886,137,69
3724887,137,162


In [166]:
taxi_zones


,LocationID,Borough,Zone
0,1,EWR,Newark Airport
1,2,Queens,Jamaica Bay
2,3,Bronx,Allerton/Pelham Gardens
3,4,Manhattan,Alphabet City
4,5,Staten Island,Arden Heights
...,...,...,...
260,261,Manhattan,World Trade Center
261,262,Manhattan,Yorkville East
262,263,Manhattan,Yorkville West
263,264,Unknown,NaN


In [146]:
#drop un needed from taxi zones
taxi_zones = taxi_zones.drop(columns=["service_zone"])
taxi_zones

KeyError: "['service_zone'] not found in axis"

In [147]:
pickup_zones = taxi_zones.rename(columns={
    "LocationID": "PULocationID",
    "Borough": "pickup_borough",
    "Zone": "pickup_zone",
})

dropoff_zones = taxi_zones.rename(columns={
    "LocationID": "DOLocationID",
    "Borough": "dropoff_borough",
    "Zone": "dropoff_zone",
})

In [148]:
yellow_taxi = yellow_taxi.merge(
    pickup_zones,
    on="PULocationID",
    how="left"
)

yellow_taxi = yellow_taxi.merge(
    dropoff_zones,
    on="DOLocationID",
    how="left"
)

In [170]:

yellow_taxi[["PULocationID","DOLocationID", "pickup_borough", "pickup_zone", "dropoff_borough", "dropoff_zone",]]


,PULocationID,DOLocationID,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone
0,239,238,Manhattan,Upper West Side South,Manhattan,Upper West Side North
1,163,162,Manhattan,Midtown North,Manhattan,Midtown East
2,43,237,Manhattan,Central Park,Manhattan,Upper East Side South
3,142,209,Manhattan,Lincoln Square East,Manhattan,Seaport
4,88,144,Manhattan,Financial District South,Manhattan,Little Italy/NoLiTa
...,...,...,...,...,...,...
3724884,237,161,Manhattan,Upper East Side South,Manhattan,Midtown Center
3724885,42,42,Manhattan,Central Harlem North,Manhattan,Central Harlem North
3724886,137,69,Manhattan,Kips Bay,Bronx,East Concourse/Concourse Village
3724887,137,162,Manhattan,Kips Bay,Manhattan,Midtown East


new columns for quick analysis

In [150]:
# new column for pickup day of week
yellow_taxi["pickup_day_of_week"] = yellow_taxi["pickup_datetime"].dt.day_name()

# new column for pickup month of year
yellow_taxi["pickup_month_of_year"] = yellow_taxi["pickup_datetime"].dt.month_name()

In [159]:
#new column for trip duration minutes
yellow_taxi["trip_duration_minutes"] = (( yellow_taxi["dropoff_datetime"]- yellow_taxi["pickup_datetime"]).dt.total_seconds()/ 60).round(2)



#new column for average speed mph
trip_duration_hours = ((yellow_taxi["dropoff_datetime"]- yellow_taxi["pickup_datetime"]).dt.total_seconds()/ 3600)
yellow_taxi["average_speed_mph"] = ((yellow_taxi["trip_distance_miles"] / trip_duration_hours )).round(2).astype(str)+ " mph"


In [160]:
yellow_taxi

,VendorID,pickup_datetime,dropoff_datetime,passenger_count,trip_distance_miles,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,...,Airport_fee,cbd_congestion_fee,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone,pickup_day_of_week,pickup_month_of_year,trip_duration_minutes,average_speed_mph
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,239,238,1,7.20,...,0.0,0.00,Manhattan,Upper West Side South,Manhattan,Upper West Side North,Thursday,January,5.55,10.49 mph
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,NaN,0.90,1.0,163,162,2,7.90,...,0.0,0.75,Manhattan,Midtown North,Manhattan,Midtown East,Thursday,January,5.72,9.45 mph
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,NaN,1.40,1.0,43,237,1,10.70,...,0.0,0.75,Manhattan,Central Park,Manhattan,Upper East Side South,Thursday,January,8.88,9.46 mph
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,142,209,1,38.70,...,0.0,0.75,Manhattan,Lincoln Square East,Manhattan,Seaport,Thursday,January,42.80,7.82 mph
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,NaN,2.16,1.0,88,144,1,13.50,...,0.0,0.75,Manhattan,Financial District South,Manhattan,Little Italy/NoLiTa,Thursday,January,13.50,9.6 mph
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724884,2,2026-01-31 23:26:00,2026-01-31 23:39:16,NaN,1.62,NaN,237,161,0,17.09,...,NaN,0.75,Manhattan,Upper East Side South,Manhattan,Midtown Center,Saturday,January,13.27,7.33 mph
3724885,2,2026-01-31 23:33:53,2026-01-31 23:34:07,NaN,0.00,NaN,42,42,0,23.19,...,NaN,0.00,Manhattan,Central Harlem North,Manhattan,Central Harlem North,Saturday,January,0.23,0.0 mph
3724886,2,2026-01-31 23:40:23,2026-01-31 23:56:10,NaN,6.84,NaN,137,69,0,29.21,...,NaN,0.75,Manhattan,Kips Bay,Bronx,East Concourse/Concourse Village,Saturday,January,15.78,26.0 mph
3724887,2,2026-01-31 23:10:21,2026-01-31 23:20:00,NaN,1.53,NaN,137,162,0,19.19,...,NaN,0.75,Manhattan,Kips Bay,Manhattan,Midtown East,Saturday,January,9.65,9.51 mph


In [161]:
yellow_taxi[yellow_taxi["average_speed_mph"].isna()]

,VendorID,pickup_datetime,dropoff_datetime,passenger_count,trip_distance_miles,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,...,Airport_fee,cbd_congestion_fee,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone,pickup_day_of_week,pickup_month_of_year,trip_duration_minutes,average_speed_mph
7134,1,2026-01-01 01:11:29,2026-01-01 01:11:29,1.0,0.0,1.0,158,264,2,15.60,...,0.0,0.75,Manhattan,Meatpacking/West Village West,Unknown,NaN,Thursday,January,0.0,NaN
9830,7,2026-01-01 01:32:18,2026-01-01 01:32:18,1.0,0.0,5.0,4,4,1,30.00,...,0.0,0.75,Manhattan,Alphabet City,Manhattan,Alphabet City,Thursday,January,0.0,NaN
11546,1,2026-01-01 02:36:53,2026-01-01 02:36:53,1.0,0.0,1.0,48,264,2,35.53,...,0.0,0.00,Manhattan,Clinton East,Unknown,NaN,Thursday,January,0.0,NaN
13379,1,2026-01-01 02:12:10,2026-01-01 02:12:10,1.0,0.0,1.0,42,264,2,16.30,...,0.0,0.00,Manhattan,Central Harlem North,Unknown,NaN,Thursday,January,0.0,NaN
16440,1,2026-01-01 03:20:23,2026-01-01 03:20:23,NaN,0.0,99.0,74,264,3,20.00,...,0.0,0.00,Manhattan,East Harlem North,Unknown,NaN,Thursday,January,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3582741,1,2026-01-29 12:58:44,2026-01-29 12:58:44,NaN,0.0,NaN,74,35,0,82.25,...,NaN,0.75,Manhattan,East Harlem North,Brooklyn,Brownsville,Thursday,January,0.0,NaN
3606392,1,2026-01-29 22:47:39,2026-01-29 22:47:39,NaN,0.0,NaN,164,164,0,24.60,...,NaN,0.75,Manhattan,Midtown South,Manhattan,Midtown South,Thursday,January,0.0,NaN
3644666,2,2026-01-30 18:55:00,2026-01-30 18:55:00,NaN,0.0,NaN,229,229,0,28.69,...,NaN,0.75,Manhattan,Sutton Place/Turtle Bay North,Manhattan,Sutton Place/Turtle Bay North,Friday,January,0.0,NaN
3662671,2,2026-01-30 22:31:00,2026-01-30 22:31:00,NaN,0.0,NaN,161,161,0,20.16,...,NaN,0.75,Manhattan,Midtown Center,Manhattan,Midtown Center,Friday,January,0.0,NaN


In [158]:
yellow_taxi[
    yellow_taxi["average_speed_mph"].isna()
    & (yellow_taxi["trip_distance_miles"] == 0)
].shape[0]

21913

In [169]:
yellow_taxi.isnull().sum()

VendorID                       0
pickup_datetime                0
dropoff_datetime               0
passenger_count          1102845
trip_distance_miles            0
RatecodeID               1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra_charges                  0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
pickup_borough              1539
pickup_zone                 4391
dropoff_borough            16286
dropoff_zone                6016
pickup_day_of_week             0
pickup_month_of_year           0
trip_duration_minutes          0
average_speed_mph           1164
dtype: int64

DATA LAKE AWS
